# Generate pre-computed image embeddings for faster training/evals

In [1]:
import pandas as pd
import os
import torchvision.transforms as T
import rasterio

import numpy as np
import timm
import torch
import torchgeo.models
from torchgeo.models import ResNet18_Weights, ResNet50_Weights, ViTSmall16_Weights
from tqdm import tqdm

In [2]:
# Configuration Options

root_dir = "/mnt/DATA/leca5365/satclip-s2-1M-2.0/"  # Root directory where the dataset is stored
dataset_csv = "full-index.csv"  # Path to the CSV file containing the dataset

encoder_model_name = "moco_vit16"
crop_size = 224  # Input image size for the encoder model
embed_dim = 256

In [3]:
def load_image(image_path):
    """Load an image from disk and return it as a numpy array."""
    try:
        with rasterio.open(image_path) as f:
            data = f.read().astype(np.float32)
    except Exception as e:
        print(f"Error loading image {image_path}: {e}")
        return None
    return data

def load_encoder(encoder_model_name):
    """Load the specified encoder model."""
    if encoder_model_name == "moco_vit16":
        # Load the MoCo ViT16 model
        print('using pretrained moco vit16')
        weights = ViTSmall16_Weights.SENTINEL2_ALL_MOCO
        in_chans = weights.meta["in_chans"]
        visual = timm.create_model("vit_small_patch16_224", in_chans=in_chans, num_classes=embed_dim)
        visual.load_state_dict(weights.get_state_dict(progress=True), strict=False)
        visual.requires_grad_(False)
        visual.head.requires_grad_(True)
        visual.eval()
        return visual
    else:
        raise ValueError(f"Unsupported encoder model: {encoder_model_name}")

def get_transform(encoder_model_name):
    """Get the appropriate transform for the specified encoder model."""
    if encoder_model_name == "moco_vit16":
        # Define the transform for MoCo ViT16
        def transform(image):
            B10 = np.zeros((1, *image.shape[1:]), dtype=image.dtype)
            image = np.concatenate([image[:10], B10, image[10:]], axis=0)
            image = torch.tensor(image)

            augmentation = T.Compose([
                T.CenterCrop((crop_size, crop_size)),
            ])
            return augmentation(image)
    else:
        raise ValueError(f"Unsupported encoder model: {encoder_model_name}")
    return transform


In [ ]:
# Load the encoder and dataset
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

encoder = load_encoder(encoder_model_name).to(device)
transform = get_transform(encoder_model_name)

df = pd.read_csv(os.path.join(root_dir, dataset_csv))

image_paths = []
embeddings = []
batch_size = 32

# Iterate through the dataset and compute embeddings for each image
# Modify to use batch processing for efficiency
for index, row in tqdm(df.iterrows(), total=df.shape[0], desc="Computing embeddings"):
    if index > 4:
        break  # Limit to first 5 images for testing
    image_path = os.path.join(root_dir, "images", row['fn'])
    image = transform(load_image(image_path)).unsqueeze(0).to(device)  # Add batch dimension
    embedding = encoder(image)

    embeddings.append(embedding.cpu().detach().numpy())
    image_paths.append(image_path)

# Save embeddings to disk as a parquet file
embeddings_array = np.vstack(embeddings)
df_embeddings = pd.DataFrame({'image_path': image_paths, 'embedding': embeddings_array.tolist()})
df_embeddings.to_parquet(os.path.join(root_dir, f"{encoder_model_name}_embeddings_TEST.parquet"), compression='snappy', index=False)


using pretrained moco vit16


Computing embeddings: 100%|██████████| 1240639/1240639 [2:59:06<00:00, 115.44it/s] 


ArrowInvalid: ('Can only convert 1-dimensional array values', 'Conversion failed for column embedding with type object')

In [15]:
dft = pd.read_parquet(os.path.join(root_dir, f"{encoder_model_name}_embeddings.parquet"))

In [19]:
len(dft.iloc[0,1])

256